# Valo Upscale GPU — Kaggle Notebook

Bản port Python của app **Valo Upscale** (Real-ESRGAN cho ảnh/video).

**Chuẩn bị:**
1. Cột phải → *Settings → Accelerator = **GPU T4 x2*** (hoặc P100).
2. Cột phải → *Settings → **Internet = On*** (bắt buộc: cần tải weight model + clone repo).
3. Sửa `REPO_URL` ở ô dưới thành link repo GitHub của bạn → **Run All**.

Khác Colab: dữ liệu vào nằm ở `/kaggle/input/...` (dataset bạn add), kết quả ghi vào `/kaggle/working/`
và chỉ được lưu khi bạn bấm **Save version** (dùng menu *Output* để tải file về).

In [ ]:
#@title 1) Cấu hình repo
REPO_URL = "https://github.com/CHANGE_ME/valo-upscale-gpu.git"  #@param {type:"string"}
REPO_DIR = "/kaggle/working/valo-upscale-gpu"  #@param {type:"string"}
print("Repo :", REPO_URL)
print("Thư mục:", REPO_DIR)

In [ ]:
#@title 2) Kiểm tra GPU
!nvidia-smi || echo "Chưa bật GPU: Settings > Accelerator > GPU T4 x2"
import platform, sys
print(platform.platform())
print(sys.version)

In [ ]:
import os, shutil, subprocess

if "CHANGE_ME" in REPO_URL:
    raise SystemExit("⚠️ Hãy sửa REPO_URL ở ô 1 thành link repo GitHub của bạn trước khi chạy.")

#@title 3) Clone repo + cài thư viện
import os, shutil, subprocess

print("ffmpeg:", shutil.which("ffmpeg"))

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True).stdout)
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!git log --oneline -1

In [ ]:
#@title 4) Kiểm tra môi trường
import subprocess, torch, valo_upscale

print("valo-upscale-gpu", valo_upscale.__version__)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])
!python -m valo_upscale.cli --list-models

## 5) Chạy thử với file mẫu trong repo

In [ ]:
#@title 5a) Ảnh mẫu: 4x
!python -m valo_upscale.cli samples/sample_lowres.png -s 4

from pathlib import Path
from IPython.display import display, Image as IPyImage
before = Path("samples/sample_lowres.png")
after = max(Path("outputs").glob("*_upscaled_4x_*.png"), key=lambda p: p.stat().st_mtime)
print(f"Trước: {before.name} | Sau: {after.name} ({after.stat().st_size // 1024} KB)")
display(IPyImage(str(before), width=420))
display(IPyImage(str(after), width=420))

In [ ]:
#@title 5b) Video mẫu: 2x, giữ nguyên audio
!python -m valo_upscale.cli samples/sample_clip.mp4 -s 2

from pathlib import Path
from IPython.display import Video, display
out = max(Path("outputs").glob("*_upscaled_2x_*.mp4"), key=lambda p: p.stat().st_mtime)
print(f"{out} — {out.stat().st_size // 1024} KB")
display(Video(str(out), embed=True, width=480))

## 6) Upscale dữ liệu bạn đã add vào notebook

Dataset Kaggle được mount ở `/kaggle/input/<ten-dataset>/`. Ô dưới quét toàn bộ ảnh/video trong đó
và upscale 4x (giới hạn 10 file mỗi lần cho nhanh; bỏ `[:10]` nếu muốn chạy hết).

In [ ]:
#@title 6a) Quét /kaggle/input và chạy
import subprocess
from pathlib import Path
from valo_upscale.probe import IMAGE_EXT, VIDEO_EXT

media = [p for p in sorted(Path("/kaggle/input").rglob("*")) if p.suffix.lower() in set(IMAGE_EXT) | set(VIDEO_EXT)]
print(f"Tìm thấy {len(media)} file:")
for p in media[:10]:
    print("  ", p)

for path in media[:10]:
    print("=" * 70)
    print("Xử lý:", path)
    subprocess.run(["python", "-m", "valo_upscale.cli", str(path), "-s", "4", "-o", "outputs", "--tile", "400"], check=False)

In [ ]:
#@title 6b) Xem & liệt kê kết quả trong /kaggle/working
from pathlib import Path
from IPython.display import Video, Image as IPyImage, display

results = sorted(Path("outputs").glob("*_upscaled_*"), key=lambda p: p.stat().st_mtime)[-6:]
for path in results:
    print(f"{path.name} — {path.stat().st_size / 1024:.0f} KB")
    display(Video(str(path), embed=True, width=420) if path.suffix == ".mp4" else IPyImage(str(path), width=420))

print("\nMẹo: bấm 'Save version' ở góc phải để lưu /kaggle/working (menu Output sẽ cho tải file về).")

## 7) Giao diện web Gradio (tuỳ chọn, cần Internet = On)

Kaggle không có `files.upload()`, nhưng Gradio `share=True` tạo link công khai *gradio.live* để bạn
kéo–thả file từ máy. Ô này chạy cho tới khi bấm **Stop**.

In [ ]:
#@title 7) Mở UI Gradio
!pip install -q -r requirements-ui.txt

import valo_upscale.app as app_module
demo = app_module.build_demo(out_dir="outputs")
demo.queue().launch(share=True, debug=False)

## 8) Mẹo & thông số

| Việc cần làm | Tham số |
|---|---|
| Phóng 2x / 3x / 4x / 6x / 8x | `-s 2`, `-s 3`, ... |
| Ảnh chụp / phong cảnh | `-n realesrgan-x4plus` |
| Anime, manga, 2D | `-n realesrgan-x4plus-anime` |
| Video | `-n realesr-animevideov3` (mặc định cho video) |
| 2x thuần AI cho ảnh | `-n realesrgan-x2plus` |
| Hết VRAM | `--tile 400` / `--tile 200` |
| Không cần AI | `-e ffmpeg_hq` (nhanh, chạy CPU) |

* Video bị clamp tối đa 3840×2160 (ngang) / 2160×3840 (dọc) như app gốc.
* Weight model tải ở lần chạy đầu và nằm trong `/root/.cache/valo-upscale/weights` (mất khi hết phiên);
  muốn giữ lại thì set `VALO_WEIGHTS_DIR=/kaggle/working/weights` trước khi chạy.
* GPU Kaggle T4 x2: chỉ GPU 0 được dùng cho suy luận (đủ nhanh); muốn chọn GPU khác thì
  đặt `CUDA_VISIBLE_DEVICES=1` trước lệnh.